# Generación de texto en español para aplicaciones empresariales con T5 (mT5) y GPT-2

**Curso:** Procesamiento de Lenguaje Natural · Unidad 5 — Modelos de lenguaje y generación de texto  
**Institución:** Universidad Tecnológica de Bolívar  
**Referencia:** Jurafsky, D. & Martin, J. H. (2025). *Speech and Language Processing* (3.ª ed.), Cap. 9–10 — Transformers y Large Language Models; Cap. 13 — Machine Translation (codificador–decodificador)  
**Compatibilidad con Python:** 3.10 · 3.11 · 3.12 · 3.13

## Contexto y Propósito

Este notebook usa dos modelos de lenguaje preentrenados **en español** para **generar texto** en situaciones de empresa: **mT5** (versión multilingüe de T5, codificador–decodificador), ajustado para resumir noticias en español, para resumir un informe financiero; y **GPT-2** (solo decodificador) entrenado con textos en español de la Biblioteca Nacional de España, para redactar descripciones de productos. Muestra cómo influyen la longitud máxima, la **temperatura** y las estrategias de decodificación (voraz, búsqueda en haz, *top-k* y *top-p*) en el resultado, qué pasa cuando se le pide a un modelo una tarea para la que no fue entrenado, y cómo medir automáticamente la calidad de un resumen con BLEU y ROUGE.

**Objetivo:** entender cómo se controla la generación de texto con modelos preentrenados y cuáles son sus límites, para decidir cuándo basta con un modelo genérico y cuándo hay que ajustarlo o usar un modelo más capaz.

> ℹ️ Versiones anteriores de este notebook usaban `t5-small` y `gpt2`, entrenados casi solo en inglés: con textos en español producían salidas pobres. Por eso se usan modelos entrenados en español.

## Ejemplos de Aplicación en el Mundo Real

Cada caso muestra, paso a paso, cómo se aplica la técnica de este notebook a un problema real.

### Finanzas y servicio al cliente — Resúmenes de informes

- **Problema:** la junta y los analistas necesitan resúmenes de distinta longitud de cada informe trimestral.
- **Entrada:** texto del informe.
- **Cómo se aplica la generación con mT5:**
  1. Se usa un modelo T5 ya ajustado para resumir en español (mT5 ajustado con MLSUM), que recibe el texto sin prefijo.
  2. Se usa búsqueda en haz (`num_beams=4`), determinista: el mismo informe produce siempre el mismo resumen.
  3. `max_length` se ajusta por audiencia (una línea o un párrafo).
- **Salida:** resumen corto y resumen largo.
- **Decisión:** las cifras se verifican contra el informe antes de publicar.

### Turismo y hospitalidad — Descripciones de alojamientos

- **Problema:** redactar descripciones para hoteles nuevos sin que todas suenen igual.
- **Entrada:** características del hotel.
- **Cómo se aplica la generación con GPT-2:**
  1. Se escribe un *prompt* con las características del hotel.
  2. Se usa muestreo (`do_sample=True`) con temperatura 0.7–0.9 y *top-p* para lograr variedad sin incoherencias.
  3. Se generan varios borradores con semillas distintas.
- **Salida:** tres borradores de descripción.
- **Decisión:** un editor elige uno y lo corrige antes de publicar.

## Requisitos Previos

**Instalación** (una sola vez, desde la terminal; en Jupyter o Colab antepón `!`):

```bash
pip install transformers torch evaluate sacrebleu rouge-score sentencepiece protobuf
```

**Librerías que se importan:**

| Librería | Para qué se usa |
|---------|-----------------|
| `transformers` | `AutoTokenizer`, `AutoModelForSeq2SeqLM` (mT5) y `AutoModelForCausalLM` (GPT-2) |
| `torch` | Semilla para que el muestreo sea reproducible |
| `evaluate` | BLEU (`sacrebleu`) y ROUGE |
| `re` | Tokenizador de ROUGE que conserva tildes y ñ |

**Modelos** (se descargan de Hugging Face la primera vez):

| Modelo | Arquitectura | Entrenamiento | Tamaño aprox. | Licencia |
|---|---|---|---|---|
| [`LeoCordoba/mt5-small-mlsum`](https://huggingface.co/LeoCordoba/mt5-small-mlsum) | mT5-small (codificador–decodificador) | Ajustado para resumir con la parte en español de MLSUM (noticias) | 1.2 GB | Apache 2.0 |
| [`PlanTL-GOB-ES/gpt2-base-bne`](https://huggingface.co/PlanTL-GOB-ES/gpt2-base-bne) | GPT-2 base (117 M parámetros) | 570 GB de texto en español de la Biblioteca Nacional de España | 0.5 GB | Apache 2.0 |

`sentencepiece` y `protobuf` los necesita el tokenizador de mT5.

## Marco metodológico: pipeline ML / LLM (4×3)

![Pipeline ML y LLM](img/Pipeline_ML_LLM_4x3.png)

El diagrama tiene una franja común de datos (arriba), dos rutas de modelado y una franja de operación (abajo): **ruta clásica** (*feature-based ML*) y **ruta LLM** (*foundation models*). Este notebook recorre la **ruta LLM / fundacional**. Etapas del diagrama que cubre:

| Etapa del diagrama | Dónde se aborda en este notebook |
|--------------------|----------------------------------|
| Foundation / Pre-trained LLM | Sección 2: mT5-small (ajustado para resumir) y GPT-2 en español |
| Prompt Engineering / In-Context Learning | Secciones 3–5: *prompts* para GPT-2 e instrucción inventada para mT5 |
| LLM Outputs | Secciones 3–6: resúmenes, descripciones, respuestas y comparación de decodificación |
| Evaluation & QA | Sección 7: BLEU y ROUGE frente a un resumen de referencia |

No incluye *fine-tuning* (PEFT) ni RAG: la sección 5 muestra precisamente por qué, para una tarea nueva, un modelo pequeño sin ajuste no basta.

## Estándares de referencia

Prácticas de este notebook alineadas con estándares de IEEE SA, ISO/IEC y la Open Source Initiative (OSI). Se citan como guía de buenas prácticas, no como certificación de cumplimiento.

| Estándar | Cómo se aplica aquí |
|----------|---------------------|
| **IEEE 7001-2021** — Transparencia de sistemas autónomos | Se documenta cómo cada parámetro de decodificación cambia la salida, y se fija la semilla para que los ejemplos con muestreo sean reproducibles. |
| **IEEE 3119-2025** — Adquisición (*procurement*) de IA y sistemas de decisión automatizada | Antes de adoptar un modelo generativo hay que probarlo en la tarea real; la sección 5 muestra un caso en que el modelo no sirve sin ajuste. |
| **ISO/IEC 23894:2023** — Guía para la gestión de riesgos de IA | Riesgos documentados: alucinaciones en los textos generados, salidas no deterministas con muestreo y métricas que no verifican hechos. |
| **IEEE 7000-2021** — Proceso modelo para abordar consideraciones éticas durante el diseño de sistemas | Los textos generados para clientes o pacientes deben pasar por revisión humana; el notebook lo indica en cada caso de uso. |
| **OSI · Open Source AI Definition 1.0 (2024)** — Qué hace que un sistema de IA sea de código abierto: libertades de usar, estudiar, modificar y compartir, con acceso a código, parámetros e información de los datos | mT5-small-mlsum y GPT-2 BNE publican pesos y código con licencia Apache 2.0, y se ejecutan localmente. |

## Objetivos de aprendizaje

Al finalizar este notebook podrás:

1. Cargar y comparar **mT5** (codificador–decodificador) y **GPT-2** (solo decodificador) en español desde el Hub de Hugging Face
2. Aplicar mT5 al **resumen automático de informes** con distintas longitudes
3. Usar GPT-2 para **generar descripciones de productos** y controlar la creatividad con `temperature`
4. Reconocer qué ocurre cuando se pide a un modelo una tarea para la que **no** fue entrenado
5. Comparar estrategias de decodificación: **voraz**, **búsqueda en haz**, **top-k** y **top-p**
6. Combinar generación y evaluación con BLEU/ROUGE en un flujo completo

## Comparación de arquitecturas

| Modelo | Arquitectura | Estilo de tarea | Formato de entrada |
|---|---|---|---|
| **mT5-small-mlsum** | Codificador–decodificador | Texto a texto; ajustado solo para resumir | Texto completo a resumir |
| **GPT-2 BNE** | Solo decodificador | Continuación autorregresiva | Texto libre (*prompt*) |

> ℹ️ El informe financiero, la empresa *Banco del Litoral* y las consultas de clientes son **ejemplos inventados**; no describen hechos reales.

In [ ]:
#!pip install evaluate

---

## 1. Configuración del entorno

In [1]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
)
import evaluate
import torch

torch.manual_seed(42)   # hace reproducibles los ejemplos con muestreo

---

## 2. Cargar mT5 y GPT-2 en español

Los modelos se descargan del Hub de Hugging Face la primera vez y quedan en caché local. mT5-small tiene ~300 millones de parámetros (su vocabulario multilingüe es grande); GPT-2 base, 117 millones.

In [ ]:
T5_MODEL = 'LeoCordoba/mt5-small-mlsum'     # mT5-small ajustado para resumir en español
GPT2_MODEL = 'PlanTL-GOB-ES/gpt2-base-bne'   # GPT-2 entrenado con textos en español

# mT5: codificador–decodificador
t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL, use_fast=False)
t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL)

# GPT-2: solo decodificador
gpt2_tokenizer = AutoTokenizer.from_pretrained(GPT2_MODEL)
gpt2_model = AutoModelForCausalLM.from_pretrained(GPT2_MODEL)

print(f'mT5 cargado:   {T5_MODEL}')
print(f'GPT-2 cargado: {GPT2_MODEL}')

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Error in sys.excepthook:
Traceback (most recent call last):
  File "C:\Users\epuerta\AppData\Local\anaconda3\Lib\site-packages\ipykernel\iostream.py", line 142, in _event_pipe
    event_pipe = self._local.event_pipe
                 ^^^^^^^^^^^^^^^^^^^^^^
AttributeError: '_thread._local' object has no attribute 'event_pipe'

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "C:\Users\epuerta\AppData\Local\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py", line 2112, in excepthook
    self.showtraceback((etype, value, tb), tb_offset=0)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\epuerta\AppData\Local\anaconda3\Lib\site-packages\IPython\core\interactiveshell.py", line 2234, in showtraceback
    self._showtraceback(etype, value, stb)
    ~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^
  File "C:\Users\epuerta\AppData\Local\anaconda3\Lib\site-packages\ipykernel\zmqshell.py", line 606, in _showtrace

> **Interpretación de la salida:** ambos modelos quedan en `~/.cache/huggingface/hub`. mT5 ocupa ~1.2 GB y GPT-2 ~0.5 GB; las ejecuciones siguientes usan la caché. En CPU la generación es más lenta; si hay GPU, mueve los modelos y las entradas con `.to("cuda")`.

---

## 3. Caso 1 — Resumen automático de informes con mT5

**Escenario:** la dirección necesita resúmenes breves de los informes financieros trimestrales. A diferencia del T5 original, que activaba el resumen con el prefijo `summarize: `, este mT5 se ajustó **solo** para resumir noticias en español, así que recibe el texto directamente.

In [ ]:
# Informe ficticio de ejemplo
financial_report = (
    'En el tercer trimestre de 2024, Banco del Litoral reportó una utilidad neta de 2,3 billones de pesos, '
    'un 15 % más que en el mismo periodo de 2023. '
    'La cartera de créditos creció un 8 %, impulsada sobre todo por los segmentos comercial e hipotecario. '
    'La cartera vencida se mantuvo estable en 3,2 %, por debajo del promedio del sector. '
    'Los canales digitales procesaron el 78 % de las transacciones, frente al 65 % del año anterior. '
    'La junta aprobó un dividendo de 250 pesos por acción, pagadero en diciembre de 2024.'
)

# mT5 ajustado para resumir: el texto entra sin prefijo de tarea
inputs = t5_tokenizer(financial_report, return_tensors='pt', max_length=512, truncation=True)

# Búsqueda en haz con 4 candidatos; resumen de 20 a 80 tokens
output_ids = t5_model.generate(
    **inputs,
    max_length=80,
    min_length=20,
    num_beams=4,
    early_stopping=True,
)
summary = t5_tokenizer.decode(output_ids[0], skip_special_tokens=True)

print(f'Original ({len(financial_report.split())} palabras):')
print(financial_report)
print(f'\nResumen ({len(summary.split())} palabras):')
print(summary)

> **Interpretación de la salida:** la búsqueda en haz (`num_beams=4`) mantiene 4 secuencias candidatas en paralelo y se queda con la más probable; suele producir resúmenes más fluidos que la decodificación voraz. `min_length=20` evita un resumen trivialmente corto. Compara el número de palabras del original y del resumen para ver cuánto condensa el modelo. **Revisa las cifras del resumen**: un modelo pequeño puede copiar mal un número o mezclar datos. Como se ajustó con noticias (MLSUM), tiende a escribir en estilo periodístico, a veces como un titular.

### 3.1 Efecto de la longitud del resumen

Variar `max_length` produce resúmenes con distinto nivel de detalle, útil para adaptar la salida a distintas audiencias (una línea para la junta, un párrafo para analistas).

In [ ]:
# Mismo informe, tres longitudes máximas (en tokens)
for max_len in (30, 60, 100):
    ids = t5_model.generate(
        **inputs, max_length=max_len, num_beams=4, early_stopping=True
    )
    text = t5_tokenizer.decode(ids[0], skip_special_tokens=True)
    print(f'max_length={max_len}: {text}')
    print()

> **Interpretación de la salida:** con un `max_length` corto el modelo debe ser más selectivo y conserva solo lo más relevante. Con uno largo hay más detalle, pero puede repetir información si el texto original ya es corto. Ojo: `max_length` es un **tope**, no un objetivo; el modelo puede terminar antes.

---

## 4. Caso 2 — Descripciones de productos con GPT-2

**Escenario:** un equipo de comercio electrónico quiere generar descripciones variadas de productos a partir de un *prompt*. El modo de muestreo de GPT-2 (`do_sample=True`) produce salidas distintas en cada ejecución.

In [ ]:
product_prompt = (
    'Producto: audífonos inalámbricos con cancelación de ruido\n'
    'Descripción:'
)

# GPT-2 no tiene token de relleno: se usa el de fin de texto para evitar advertencias
gpt2_tokenizer.pad_token = gpt2_tokenizer.eos_token
inputs_gpt2 = gpt2_tokenizer(product_prompt, return_tensors='pt')
n_prompt = inputs_gpt2['input_ids'].shape[1]   # para mostrar solo el texto nuevo

# Muestreo: temperatura 0.8 entre los 50 tokens más probables, hasta 60 tokens nuevos
output_ids = gpt2_model.generate(
    **inputs_gpt2,
    max_new_tokens=60,
    do_sample=True,
    temperature=0.8,
    top_k=50,
    pad_token_id=gpt2_tokenizer.eos_token_id,
)
description = gpt2_tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(description)

> **Interpretación de la salida:** GPT-2 continúa el *prompt* palabra a palabra. La temperatura reescala las probabilidades antes de muestrear: con 1.0 se usan tal cual; por debajo de 1 se concentran en los tokens más probables (más conservador) y por encima de 1 se aplanan (más variado y más errático). `top_k=50` limita el muestreo a los 50 tokens más probables en cada paso, evitando elecciones muy improbables. GPT-2 base no sigue instrucciones: es frecuente que la “descripción” se desvíe del producto. Que esté entrenado en español mejora la fluidez, no la obediencia.

### 4.1 Efecto de la temperatura en la diversidad

In [ ]:
# Mismo prompt con tres temperaturas
for temp in (0.3, 0.7, 1.2):
    ids = gpt2_model.generate(
        **inputs_gpt2,
        max_new_tokens=40,
        do_sample=True,
        temperature=temp,
        top_k=50,
        pad_token_id=gpt2_tokenizer.eos_token_id,
    )
    text = gpt2_tokenizer.decode(ids[0][n_prompt:], skip_special_tokens=True)   # solo el texto generado
    print(f'temperature={temp}:')
    print(text)
    print()

> **Interpretación de la salida:** con temperatura baja (0.3) la salida es casi determinista y tiende a repetirse: segura pero monótona. Con temperatura alta (1.2) aparecen frases novedosas, pero también incoherencias y errores. Para textos comerciales que deben ser variados y coherentes, suele usarse un valor intermedio (0.7–0.9).

---

## 5. Caso 3 — Respuestas de atención al cliente con mT5 (un límite del modelo)

**Escenario:** se quiere generar respuestas a consultas frecuentes de clientes con el mismo mT5, anteponiendo una instrucción inventada: `responde al cliente:`.

Este mT5 solo aprendió a **resumir**. Una instrucción nueva no le enseña otra tarea: este caso muestra qué ocurre cuando se usa un modelo para algo que no aprendió.

In [ ]:
customer_queries = [
    'No puedo ingresar a mi cuenta de banca en línea.',
    'Mi tarjeta fue rechazada en el supermercado.',
    'Quiero aumentar el cupo de mi tarjeta de crédito.',
]

# Instrucción que el modelo no vio en su entrenamiento
for query in customer_queries:
    prompt = f'responde al cliente: {query}'
    ids_in = t5_tokenizer(prompt, return_tensors='pt', max_length=128, truncation=True)
    ids_out = t5_model.generate(**ids_in, max_length=80, num_beams=4, early_stopping=True)
    response = t5_tokenizer.decode(ids_out[0], skip_special_tokens=True)
    print(f'P: {query}')
    print(f'R: {response}')
    print()

> **Interpretación de la salida:** lo más probable es que mT5 repita o reformule la consulta como si fuera un titular, en lugar de responderla: hace lo único que sabe, resumir. Para un sistema real hay tres caminos: (1) ajustar el modelo con pares `(consulta, respuesta ideal)` de los tickets de soporte; (2) usar un LLM ajustado a instrucciones (ver `prompting_techniques/`); (3) recuperar respuestas aprobadas de una base de conocimiento (RAG). En atención al cliente, toda respuesta generada debe validarse antes de enviarse.

---

## 6. Comparación de estrategias de decodificación

Cada estrategia equilibra de forma distinta la calidad, la diversidad y el costo de cómputo.

| Estrategia | Parámetro clave | Equilibrio |
|---|---|---|
| **Voraz** | — | Rápida; a menudo repetitiva |
| **Búsqueda en haz** | `num_beams` | Más fluida; menos diversa |
| **Muestreo top-k** | `top_k` | Creativa; puede ser incoherente |
| **Top-p (núcleo)** | `top_p` | Vocabulario dinámico; mejor coherencia que top-k |

In [ ]:
report_input = t5_tokenizer(
    financial_report,
    return_tensors='pt', max_length=512, truncation=True,
)

# Parámetros de generate() para cada estrategia
strategies = {
    'voraz': {'max_length': 60},
    'búsqueda en haz': {'max_length': 60, 'num_beams': 4, 'early_stopping': True},
    'top_k': {'max_length': 60, 'do_sample': True, 'top_k': 50, 'temperature': 0.8},
    'núcleo (top_p)': {'max_length': 60, 'do_sample': True, 'top_p': 0.92, 'temperature': 0.8},
}

for name, kwargs in strategies.items():
    ids = t5_model.generate(**report_input, **kwargs)
    text = t5_tokenizer.decode(ids[0], skip_special_tokens=True)
    print(f'[{name}] {text}')
    print()

> **Interpretación de la salida:** la búsqueda en haz suele dar la salida más gramatical. El muestreo por núcleo (top-p) elige entre el conjunto más pequeño de tokens cuya probabilidad acumulada supera `top_p`, de modo que el número de candidatos se adapta a cada paso; suele ser más diverso que top-k fijo sin perder coherencia. Para salidas deterministas y auditables (resúmenes legales o financieros), prefiere la búsqueda en haz; para textos creativos, el muestreo por núcleo.

---

## 7. Flujo completo — generación y evaluación automática

In [ ]:
import re

bleu_metric = evaluate.load('sacrebleu')
rouge_metric = evaluate.load('rouge')


def tokenizar_es(texto: str) -> list:
    """Tokenizador para ROUGE que conserva tildes y ñ (el de rouge-score las elimina)."""
    return re.findall(r'\w+', texto.lower())


# Resumen de referencia (escrito por una persona) del mismo informe ficticio
reference_summary = (
    'Banco del Litoral aumentó su utilidad neta un 15 % en el tercer trimestre de 2024, '
    'con el 78 % de las transacciones en canales digitales y un dividendo de 250 pesos por acción.'
)

# Resumen generado con búsqueda en haz
generated_ids = t5_model.generate(**report_input, max_length=80, num_beams=4, early_stopping=True)
generated_summary = t5_tokenizer.decode(generated_ids[0], skip_special_tokens=True)

bleu = bleu_metric.compute(predictions=[generated_summary], references=[[reference_summary]])
rouge = rouge_metric.compute(predictions=[generated_summary], references=[reference_summary],
                             tokenizer=tokenizar_es)

print(f'Generado:   {generated_summary}')
print(f'Referencia: {reference_summary}')
print(f'BLEU:    {bleu["score"]:.2f}')
print(f'ROUGE-1: {rouge["rouge1"]:.4f}')
print(f'ROUGE-L: {rouge["rougeL"]:.4f}')

> **Interpretación de la salida:** el tokenizador por defecto de `rouge-score` solo conserva letras de la *a* a la *z*: partiría *aumentó* en *aument* y *ó* y *año* en *a* y *o*. Por eso se le pasa `tokenizar_es`. BLEU y ROUGE miden el solapamiento de n-gramas entre el resumen generado y la referencia. Un puntaje bajo no implica necesariamente mala calidad (el modelo puede parafrasear bien), y uno alto no garantiza que las cifras sean correctas. Para una señal más fiable, usa varias referencias, añade BERTScore (ver `metrics_llm_business.ipynb`) y revisa a mano los datos clave. En producción, registra estas métricas por documento para detectar degradación del modelo con el tiempo.

---

## Resumen

| Tarea | Modelo | Por qué |
|---|---|---|
| Resumen de informes | mT5 ajustado para resumir | Codificador–decodificador ajustado con resúmenes de noticias en español |
| Textos comerciales creativos | GPT-2 | Autorregresivo; la temperatura controla la creatividad |
| Respuestas de atención al cliente | Modelo **ajustado** a la tarea o un LLM con instrucciones | Un modelo pequeño sin ajuste no conoce la tarea |
| Evaluación de calidad | BLEU + ROUGE + BERTScore + revisión humana | Varias métricas y verificación de hechos |